In [1]:
# --------------------- créer un fichier ------------------------

# ".": dossier courant
# "..": dossier parent
# "~" : dossier utilisateur
# mode "w" : création si le fichier existe ET écrasement du contenu s'il existe
# attention souvent sous windows/Excell on peut avoir des fichiers encodés en iso-8859-1

# creér (ouvrir en mode création) le fichier "mon_fichier.txt" en encodage utf-8
f = open("./mon_fichier.txt", mode="w", encoding="utf-8")
f.write("1ère ligne\n")
f.write("2ème ligne\n")

# important! il faut fermer ce qu'on a ouvert
f.close()


In [3]:
# ----------------- lire le contenu d'un fichier ---------------------

# ouvrir le fichier en lecture et même encodage
f = open("./mon_fichier.txt", mode="r", encoding="utf-8")
# lire une ligne (avec \n)
print(f.read(11))
# lire tout
print(f.read())
## REM. notion de curseur: la deuxième lecture reprend 
## depuis la fin de la première ligne
f.close()

1ère ligne

2ème ligne



In [4]:
# ------------- écriture à la fin du fichier (append) -------------------
# écrire une 3ème ligne dans le fichier 
f = open("./mon_fichier.txt", mode="a", encoding="utf-8")
f.write("3ème ligne\n")
# vérifier que le contenu existant n'a pas été supprimé
f.close()

# vérifier le contenu
f = open("./mon_fichier.txt", mode="r", encoding="utf-8")
print(f.read())
f.close()

1ère ligne
2ème ligne
3ème ligne



In [21]:
# --------------------- readlines / writelines --------------

# lire toutes les lignes dans une liste
f = open("./mon_fichier.txt", mode="w", encoding="utf-8")
f.writelines([
    "1ère ligne\n",
    "2ème ligne\n",
    "3ème ligne\n"
])
f.close()

# lire les lignes
f = open("./mon_fichier.txt", mode="r", encoding="utf-8")
lines = f.readlines()
print(lines)
f.close()


['1ère ligne\n', '2ème ligne\n', '3ème ligne\n']


In [18]:
# ------------------- modes avancés ------------------------------
# remplacer la ligne n
# r+: on a un curseur pour la lecture et l'écriture

# je veux intercaler une nouvelle ligne 
# après la ligne n
n = 2

with open("./mon_fichier.txt", mode="r+", encoding="utf-8") as f:
   # f.read(11)
   # par défaut l'écriture ECRASE LE CONTENU EXISTANT
   # MAIS dès qu'on a lu une fois, le curseur en écriture est à la fin => difficile à manipuler
   # f.write("nouvelle ligne\n")

   # manipuler le curseur d'écriture/lecture est dangereux car on compte ici des octets
   # or utf-8 encode certains caractères sur plusieurs octets !=
   # f.seek(13)
   # print(f"position du curseur: {f.tell()}")
   # f.write("nouvelle ligne\n")

   # positionnement à cheval sur un caractère UTF-8 multi-octets
   f.seek(2)
   f.read() # UnicodeDecodeError: 'utf-8' codec can't decode byte 0xa8 in position 0: invalid start byte

UnicodeDecodeError: 'utf-8' codec can't decode byte 0xa8 in position 0: invalid start byte

In [ ]:
# -- faciliter les ouvertures/fermetures: gestionnaires de contexte: with --

## with <ouverture>() as <var>:
  # fichier ouvert
  # ....
with open("./mon_fichier.txt", mode="r", encoding="utf-8") as f:
    lines = f.readlines()
    print(lines)

# ici fichier fermé

In [6]:
# ----------- un fichier est un itérable de lignes ----------------------

with open("./mon_fichier.txt", mode="r", encoding="utf-8") as f:
    # for line in f:
    #     print(line, end='')
    print(list(f))

['1ère ligne\n', '2ème ligne\n', '3ème ligne\n']


In [22]:
# --------- suppression de la ligne n avec for de façon HAUT NIVEAU ----------

n = 2
with open("./mon_fichier.txt", mode="r", encoding="utf-8") as f:
    lines = f.readlines()

lines = lines[:n-1] + lines[n:]

with open("./mon_fichier.txt", mode="w", encoding="utf-8") as f:
    f.writelines(lines)


# écraser le fichier avec les lignes de liste

In [23]:
# -----------------  création d'un csv -----------------------------

import csv
# importer le module standard csv de la bibliothèque standard

users = [
  {"firstname":"Joe", "lastname": "Doe", "age": 22, "height": 1.75, "comment": "blablab ... ; blabliblo"},
  {"firstname": "Jane", "lastname": "Austen", "age": 34, "height": 1.79, "comment": "blabla"},
]

header = list(users[0])

# transformer une liste de dictionnaire en liste de liste
data = list(map(lambda d: list(d.values()), users))
print(data)

# ouvrir le fichier users.csv en création en utf-8
# pb de portabilité avec csv.writer avec Windows: rajoute deux sauts de ligne
# contournement 1/ newline="" dans open
# contournement 2/ lineterminator="\n" dans csv.writer
with open("./users.csv", mode="w", encoding="utf-8", newline="") as f:
    writer = csv.writer(f, delimiter=";")
    # writer = csv.writer(f, delimiter=";", lineterminator="\n")
    writer.writerow(header)
    writer.writerows(data)

# créer un writer issu du module csv, utilisant le fichier
# utiliser un writer pour écrire le header du csv à partir des clés des dict
# //                             les données du csv //         valeurs du dict

[['Joe', 'Doe', 22, 1.75, 'blablab ... ; blabliblo'], ['Jane', 'Austen', 34, 1.79, 'blabla']]


In [ ]:
# ------------------- idem mais avec le bon outil -------------------------
import csv

users = [
  {"firstname":"Joe", "lastname": "Doe", "age": 22, "height": 1.75, "comment": "blablab ... ; blabliblo"},
  {"firstname": "Jane", "lastname": "Austen", "age": 34, "height": 1.79, "comment": "blabla"},
]

header = list(users[0])

with open("./users.csv", mode="w", encoding="utf-8", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=header, delimiter=";")
    writer.writeheader()
    writer.writerows(users)

In [24]:
# --------------- lire les lignes de csv ---------------
import csv
# idem mais en lecture
# protéger l'ouverture du fichier s'il le fichier existe
# tip: on peut itérer de façon manuelle un itérable avec la fonction next()
#    : pour récolter le header en première ligne

with open("./users.csv", mode="r", encoding="utf-8", newline="") as f:
    reader = csv.reader(f, delimiter=";")
    header = next(reader)
    users = []
    for row in reader:
        users.append(dict(zip(header, row)))
print(users)

[{'firstname': 'Joe', 'lastname': 'Doe', 'age': '22', 'height': '1.75', 'comment': 'blablab ... ; blabliblo'}, {'firstname': 'Jane', 'lastname': 'Austen', 'age': '34', 'height': '1.79', 'comment': 'blabla'}]


In [ ]:
# ----------------------- écriture en JSON ----------------------

import json
# idem avec json 
# sachant que les objets et les dictionnaires python sont très proches
# méthodes pour écrire: json.dump et json.dumps
# écrire le json de façon comprimée ou dépliée (sep, indent)

users = [
  {"firstname":"Joe", "lastname": "Doe", "age": 22, "height": 1.75, "comment": "blablab ... ; blabliblo"},
  {"firstname": "Jane", "lastname": "Austen", "age": 34, "height": 1.79, "comment": "blabla"},
]

with open("./users.json", mode="w", encoding="utf-8") as f:
    # écrire le json de façon dépliée avec indentation
    # gérer les caractères spéciaux et l'encodage
    json.dump(users, f, indent=4, ensure_ascii=False)
    # écrire de façon reserrée
    # json.dump(users, f, separators=(",", ":"))

In [ ]:
# --------------- lire un json ---------------------------
import json

with open("./users.json", mode="r", encoding="utf-8") as f:
    users = json.load(f)
    # ou si le json vient d'une str
    # users = json.loads(f.read())
print(users)
# idem en lecture avec json.load ou json.loads